In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
    accuracy_score,
)
from sklearn.calibration import CalibrationDisplay

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
DATA_DIR = '../data/raw/'           # adjust to your Kaggle data path
GENDER = 'W'                 # 'M' for mens, 'W' for womens
VAL_START_SEASON = 2016      # first season used as a validation fold

import logging
logging.getLogger('mlflow').setLevel(logging.ERROR)

from xgboost import XGBClassifier
from sklearn.model_selection import ParameterGrid

pd.set_option('display.max_columns', None)

In [2]:

EXPERIMENT_NAME = f'march_madness_{GENDER.lower()}_2026'

mlflow.set_tracking_uri(f'../mlruns')
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'MLflow experiment: {EXPERIMENT_NAME}')

MLflow experiment: march_madness_w_2026


In [4]:
prefix = GENDER  # 'M' or 'W'

tourney    = pd.read_csv(f'{DATA_DIR}{prefix}NCAATourneyCompactResults.csv')
seeds      = pd.read_csv(f'{DATA_DIR}{prefix}NCAATourneySeeds.csv')
reg_season = pd.read_csv(f'{DATA_DIR}{prefix}RegularSeasonCompactResults.csv')

try:
    reg_detailed = pd.read_csv(f'{DATA_DIR}{prefix}RegularSeasonDetailedResults.csv')
    print(f'Detailed box scores: {reg_detailed.Season.min()}–{reg_detailed.Season.max()}')
except FileNotFoundError:
    reg_detailed = None
    print('No detailed results file — basic features only.')

try:
    massey = pd.read_csv(f'{DATA_DIR}{prefix}MasseyOrdinals.csv')
    print(f'Massey ordinals: {massey.Season.min()}–{massey.Season.max()}')
except FileNotFoundError:
    massey = None
    print('No Massey ordinals file found.')

print(f'Tourney games: {len(tourney)}')
print(f'Seasons: {tourney.Season.min()}–{tourney.Season.max()}')

Detailed box scores: 2010–2026
No Massey ordinals file found.
Tourney games: 1717
Seasons: 1998–2025


## Seeds

In [5]:
# ── 4a. Parse seed number from seed string (e.g. 'W01' → 1) ──────────────────
def parse_seed(seed_str):
    return int(''.join(filter(str.isdigit, str(seed_str))))

seeds['SeedNum'] = seeds['Seed'].apply(parse_seed)
seed_lookup = seeds.set_index(['Season', 'TeamID'])['SeedNum'].to_dict()

## Elo Rating

In [6]:
# ── 4b. Elo ratings (carries across seasons with mean reversion) ──────────────
def compute_elo(results_df, k=32, initial=1500, carry_factor=0.75, use_mov=True):
    elo = {}
    season_end = {}

    for season, grp in results_df.sort_values(['Season', 'DayNum']).groupby('Season'):
        elo = {
            team: carry_factor * rating + (1 - carry_factor) * initial
            for team, rating in elo.items()
        }

        for _, row in grp.iterrows():
            w, l = row['WTeamID'], row['LTeamID']
            r_w = elo.get(w, initial)
            r_l = elo.get(l, initial)

            # Home court adjustment
            loc = row.get('WLoc', 'N')
            hca = 100
            r_w_adj = r_w + (hca if loc == 'H' else -hca if loc == 'A' else 0)

            exp_w = 1 / (1 + 10 ** ((r_l - r_w_adj) / 400))
            exp_l = 1 - exp_w

            # MOV adjusted K
            if use_mov and 'WScore' in row.index:
                mov = row['WScore'] - row['LScore']
                elo_diff = abs(r_w_adj - r_l)
                mov_mult = np.log(abs(mov) + 1.0)
                elo_mult = 2.2 / ((elo_diff * 0.001) + 2.2)
                k_adj = k * mov_mult * elo_mult
            else:
                k_adj = k

            elo[w] = r_w + k_adj * (1 - exp_w)
            elo[l] = r_l + k_adj * (0 - exp_l)

        for team, rating in elo.items():
            season_end[(season, team)] = rating

    return season_end

print('Computing Elo ratings...')
elo_ratings = compute_elo(reg_season, k=20, carry_factor=0.75)
print(f'Elo ratings computed for {len(elo_ratings)} (season, team) pairs.')

Computing Elo ratings...
Elo ratings computed for 9952 (season, team) pairs.


## Season and Last-N Advanced Metrics

In [ ]:
# ── 4c. Last-N form features from detailed box scores ────────────────────────
def compute_form_features(detailed_df, n_games=10):
    if detailed_df is None:
        return pd.DataFrame()

    records = []
    for _, row in detailed_df.iterrows():
        season = row['Season']
        day    = row['DayNum']

        w_poss = row['WFGA'] - row['WOR'] + row['WTO'] + 0.44 * row['WFTA']
        l_poss = row['LFGA'] - row['LOR'] + row['LTO'] + 0.44 * row['LFTA']
        poss   = (w_poss + l_poss) / 2 if (w_poss + l_poss) > 0 else 1

        w_off_eff = (row['WScore'] / poss) * 100
        l_off_eff = (row['LScore'] / poss) * 100

        records.append({
            'Season': season, 'DayNum': day,
            'TeamID': row['WTeamID'], 'Win': 1,
            'Margin': row['WScore'] - row['LScore'],
            'OffEff': w_off_eff, 'DefEff': l_off_eff,
            'Pace': poss
        })
        records.append({
            'Season': season, 'DayNum': day,
            'TeamID': row['LTeamID'], 'Win': 0,
            'Margin': row['LScore'] - row['WScore'],
            'OffEff': l_off_eff, 'DefEff': w_off_eff,
            'Pace': poss
        })

    game_df = pd.DataFrame(records).sort_values(['Season', 'TeamID', 'DayNum'])

    form_rows = []
    for (season, team), grp in game_df.groupby(['Season', 'TeamID']):
        last_n = grp.tail(n_games)

        # ── Season-long stats ────────────────────────────────────────────────
        season_row = {
            'Season': season,
            'TeamID': team,
            # Season long
            'Season_WinRate':   grp['Win'].mean(),
            'Season_AvgMargin': grp['Margin'].mean(),
            'Season_AvgOffEff': grp['OffEff'].mean(),
            'Season_AvgDefEff': grp['DefEff'].mean(),
            'Season_EffMargin': grp['OffEff'].mean() - grp['DefEff'].mean(),
            'Season_AvgPace':   grp['Pace'].mean(),
            'Season_NumGames':  len(grp),
            # Last N
            f'Last{n_games}_WinRate':   last_n['Win'].mean(),
            f'Last{n_games}_AvgMargin': last_n['Margin'].mean(),
            f'Last{n_games}_AvgOffEff': last_n['OffEff'].mean(),
            f'Last{n_games}_AvgDefEff': last_n['DefEff'].mean(),
            f'Last{n_games}_EffMargin': last_n['OffEff'].mean() - last_n['DefEff'].mean(),
            f'Last{n_games}_AvgPace':   last_n['Pace'].mean(),
            # Trend — last N minus season average, captures peaking/declining
            'Trend_EffMargin': (last_n['OffEff'].mean() - last_n['DefEff'].mean()) - 
                               (grp['OffEff'].mean() - grp['DefEff'].mean()),
            'Trend_WinRate':   last_n['Win'].mean() - grp['Win'].mean(),
        }
        form_rows.append(season_row)

    return pd.DataFrame(form_rows).set_index(['Season', 'TeamID'])

print('Computing form features...')
form_df = compute_form_features(reg_detailed, n_games=10)
if not form_df.empty:
    print(f'Form features: {form_df.shape}')
else:
    print('No form features (detailed data unavailable).')

Computing form features...
Form features: (5965, 15)


## Four Factors (Shooting, Turnovers, Rebounding, Free Throws)

In [9]:
def compute_four_factors(detailed_df, n_games=10):
    if detailed_df is None:
        return pd.DataFrame()

    records = []
    for _, row in detailed_df.iterrows():
        season = row['Season']
        day    = row['DayNum']

        # Possessions
        w_poss = row['WFGA'] - row['WOR'] + row['WTO'] + 0.44 * row['WFTA']
        l_poss = row['LFGA'] - row['LOR'] + row['LTO'] + 0.44 * row['LFTA']

        # Effective FG% = (FGM + 0.5 * 3PM) / FGA
        w_efg  = (row['WFGM'] + 0.5 * row['WFGM3']) / row['WFGA'] if row['WFGA'] > 0 else 0.5
        l_efg  = (row['LFGM'] + 0.5 * row['LFGM3']) / row['LFGA'] if row['LFGA'] > 0 else 0.5

        # Turnover rate = TO / possessions
        w_tor  = row['WTO'] / w_poss if w_poss > 0 else 0.15
        l_tor  = row['LTO'] / l_poss if l_poss > 0 else 0.15

        # Offensive rebound rate = ORB / (ORB + opp DRB)
        w_orb  = row['WOR'] / (row['WOR'] + row['LDR']) if (row['WOR'] + row['LDR']) > 0 else 0.3
        l_orb  = row['LOR'] / (row['LOR'] + row['WDR']) if (row['LOR'] + row['WDR']) > 0 else 0.3

        # Free throw rate = FTA / FGA
        w_ftr  = row['WFTA'] / row['WFGA'] if row['WFGA'] > 0 else 0.3
        l_ftr  = row['LFTA'] / row['LFGA'] if row['LFGA'] > 0 else 0.3

        # 3-point rate = FGA3 / FGA
        w_3pr  = row['WFGA3'] / row['WFGA'] if row['WFGA'] > 0 else 0.3
        l_3pr  = row['LFGA3'] / row['LFGA'] if row['LFGA'] > 0 else 0.3

        # 3-point % 
        w_3p   = row['WFGM3'] / row['WFGA3'] if row['WFGA3'] > 0 else 0.33
        l_3p   = row['LFGM3'] / row['LFGA3'] if row['LFGA3'] > 0 else 0.33

        records.append({
            'Season': season, 'DayNum': day,
            'TeamID': row['WTeamID'],
            'EFG': w_efg, 'EFGD': l_efg,
            'TOR': w_tor, 'TORD': l_tor,
            'ORB': w_orb, 'DRB': 1 - l_orb,
            'FTR': w_ftr, 'FTRD': l_ftr,
            'ThreePR': w_3pr, 'ThreePRD': l_3pr,
            'ThreeP': w_3p,  'ThreePD': l_3p,
        })
        records.append({
            'Season': season, 'DayNum': day,
            'TeamID': row['LTeamID'],
            'EFG': l_efg, 'EFGD': w_efg,
            'TOR': l_tor, 'TORD': w_tor,
            'ORB': l_orb, 'DRB': 1 - w_orb,
            'FTR': l_ftr, 'FTRD': w_ftr,
            'ThreePR': l_3pr, 'ThreePRD': w_3pr,
            'ThreeP': l_3p,  'ThreePD': w_3p,
        })

    game_df = pd.DataFrame(records).sort_values(['Season', 'TeamID', 'DayNum'])

    ff_rows = []
    for (season, team), grp in game_df.groupby(['Season', 'TeamID']):
        last_n = grp.tail(n_games)
        ff_rows.append({
            'Season': season, 'TeamID': team,
            # Season averages
            'Season_EFG':    grp['EFG'].mean(),
            'Season_EFGD':   grp['EFGD'].mean(),
            'Season_TOR':    grp['TOR'].mean(),
            'Season_TORD':   grp['TORD'].mean(),
            'Season_ORB':    grp['ORB'].mean(),
            'Season_DRB':    grp['DRB'].mean(),
            'Season_FTR':    grp['FTR'].mean(),
            'Season_FTRD':   grp['FTRD'].mean(),
            'Season_3PR':    grp['ThreePR'].mean(),
            'Season_3PRD':   grp['ThreePRD'].mean(),
            'Season_3P':     grp['ThreeP'].mean(),
            'Season_3PD':    grp['ThreePD'].mean(),
            # Last N averages
            f'Last{n_games}_EFG':  last_n['EFG'].mean(),
            f'Last{n_games}_EFGD': last_n['EFGD'].mean(),
            f'Last{n_games}_TOR':  last_n['TOR'].mean(),
            f'Last{n_games}_TORD': last_n['TORD'].mean(),
            f'Last{n_games}_ORB':  last_n['ORB'].mean(),
            f'Last{n_games}_3PR':  last_n['ThreePR'].mean(),
            f'Last{n_games}_3P':   last_n['ThreeP'].mean(),
        })

    return pd.DataFrame(ff_rows).set_index(['Season', 'TeamID'])

print('Computing four factors features...')
ff_df = compute_four_factors(reg_detailed, n_games=10)
print(f'Four factors features: {ff_df.shape}')

Computing four factors features...
Four factors features: (5965, 19)


In [10]:
# Compute opponent win rate lookup — used inside compute_sos_features
win_counts = reg_season.groupby(['Season', 'WTeamID']).size().reset_index()
win_counts.columns = ['Season', 'TeamID', 'Wins']

game_counts = pd.concat([
    reg_season[['Season', 'WTeamID']].rename(columns={'WTeamID': 'TeamID'}),
    reg_season[['Season', 'LTeamID']].rename(columns={'LTeamID': 'TeamID'})
]).groupby(['Season', 'TeamID']).size().reset_index()
game_counts.columns = ['Season', 'TeamID', 'Games']

win_rates = win_counts.merge(game_counts, on=['Season', 'TeamID'])
win_rates['WinRate'] = win_rates['Wins'] / win_rates['Games']
win_rate_lookup = win_rates.set_index(['Season', 'TeamID'])['WinRate'].to_dict()

print(f'Win rate lookup: {len(win_rate_lookup)} entries')

Win rate lookup: 9818 entries


## Strength of Schedule

In [11]:
def compute_sos_features(results_df, elo_ratings, win_rate_lookup, detailed_df=None, n_games=10):
    """
    Compute strength of schedule features for each (season, team).
    
    Two versions:
    - Season SOS: average opponent Elo and efficiency margin across all regular season games
    - Last-N SOS: same but only for the last n games
    
    Returns DataFrame indexed by (Season, TeamID).
    """
    records = []
    for _, row in results_df.sort_values(['Season', 'DayNum']).iterrows():
        season = row['Season']
        w_team, l_team = row['WTeamID'], row['LTeamID']
        w_elo = elo_ratings.get((season, w_team), 1500)
        l_elo = elo_ratings.get((season, l_team), 1500)

        records.append({
            'Season': season, 'DayNum': row['DayNum'],
            'TeamID': w_team, 'OppElo': l_elo,
            'OppWinRate': win_rate_lookup.get((season, l_team), 0.5),
        })
        records.append({
            'Season': season, 'DayNum': row['DayNum'],
            'TeamID': l_team, 'OppElo': w_elo,
            'OppWinRate': win_rate_lookup.get((season, w_team), 0.5),
        })

    opp_df = pd.DataFrame(records).sort_values(['Season', 'TeamID', 'DayNum'])

    # If detailed box scores available, also compute opponent efficiency margin
    if detailed_df is not None:
        # Reuse the game_df logic from compute_form_features
        eff_records = []
        for _, row in detailed_df.iterrows():
            season = row['Season']
            day    = row['DayNum']

            w_poss = row['WFGA'] - row['WOR'] + row['WTO'] + 0.44 * row['WFTA']
            l_poss = row['LFGA'] - row['LOR'] + row['LTO'] + 0.44 * row['LFTA']
            poss   = (w_poss + l_poss) / 2 if (w_poss + l_poss) > 0 else 1

            w_eff_margin = ((row['WScore'] - row['LScore']) / poss) * 100
            l_eff_margin = -w_eff_margin

            eff_records.append({
                'Season': season, 'DayNum': day,
                'TeamID': row['WTeamID'], 'OppEffMargin': l_eff_margin
            })
            eff_records.append({
                'Season': season, 'DayNum': day,
                'TeamID': row['LTeamID'], 'OppEffMargin': w_eff_margin
            })

        eff_df = pd.DataFrame(eff_records).sort_values(['Season', 'TeamID', 'DayNum'])
        opp_df = opp_df.merge(eff_df, on=['Season', 'DayNum', 'TeamID'], how='left')
    else:
        opp_df['OppEffMargin'] = np.nan

    # Compute season and last-N SOS per team
    sos_rows = []
    for (season, team), grp in opp_df.groupby(['Season', 'TeamID']):
        last_n = grp.tail(n_games)
        row = {
            'Season': season,
            'TeamID': team,
            # Season SOS
            'SOS_Season_AvgOppElo':       grp['OppElo'].mean(),
            'SOS_Season_AvgOppEffMargin': grp['OppEffMargin'].mean(),
            'SOS_Season_AvgOppWinRate':   grp['OppWinRate'].mean(),   # ← add this
            # Last-N SOS
            f'SOS_Last{n_games}_AvgOppElo':       last_n['OppElo'].mean(),
            f'SOS_Last{n_games}_AvgOppEffMargin': last_n['OppEffMargin'].mean(),
            f'SOS_Last{n_games}_AvgOppWinRate':   last_n['OppWinRate'].mean(),  # ← and this
        }
        sos_rows.append(row)

    return pd.DataFrame(sos_rows).set_index(['Season', 'TeamID'])


print('Computing SOS features...')
sos_df = compute_sos_features(reg_season, elo_ratings, win_rate_lookup, reg_detailed, n_games=10)
print(f'SOS features: {sos_df.shape}')

Computing SOS features...
SOS features: (9851, 6)


## Build Matchup Features

In [13]:
def build_matchup_features(tourney_df, elo_ratings, seed_lookup, form_df, sos_df, ff_df):
    """
    Standardized feature builder. 
    Accepts DataFrames for form, coach, sos, and ff data.
    """
    rows = []
    
    # Default dict for missing Massey data
    default_massey = {
        'RankMedian': np.nan, 'RankMean': np.nan, 
        'RankMin': np.nan, 'RankMax': np.nan, 
        'RankStd': np.nan, 'RankIQR': np.nan
    }

    for _, game in tourney_df.iterrows():
        season = int(game['Season'])
        # Handle both training data (WTeamID/LTeamID) and sub data (WTeamID/LTeamID as TeamA/TeamB)
        t1, t2 = int(game['WTeamID']), int(game['LTeamID'])

        # Logic to ensure Team A is always the lower TeamID for symmetry
        if t1 < t2:
            t_a, t_b = t1, t2
            label = 1 if 'Label' not in game or game.get('Label', 1) == 1 else 0 
        else:
            t_a, t_b = t2, t1
            label = 0 if 'Label' not in game or game.get('Label', 1) == 1 else 1

        # 1. Core Features
        seed_a = seed_lookup.get((season, t_a), 16)
        seed_b = seed_lookup.get((season, t_b), 16)
        elo_a  = elo_ratings.get((season, t_a), 1500)
        elo_b  = elo_ratings.get((season, t_b), 1500)

        row = {
            'Season': season, 'TeamA': t_a, 'TeamB': t_b, 'Label': label,
            'SeedA': seed_a, 'SeedB': seed_b, 'SeedDiff': seed_a - seed_b,
            'EloA': elo_a,   'EloB': elo_b,   'EloDiff':  elo_a - elo_b,
        }

# 2. Dynamic Feature Adder (Fixed to prevent double _Diff)
        def add_stats(target_row, df, prefix):
            if df is not None and hasattr(df, 'columns'):
                for col in df.columns:
                    if df[col].dtype == 'object':
                        continue
                    # If the column already ends in _Diff, don't add it again
                    feature_name = col if col.endswith('_Diff') else f'{col}_Diff'
                    
                    val_a = df.loc[(season, t_a), col] if (season, t_a) in df.index else np.nan
                    val_b = df.loc[(season, t_b), col] if (season, t_b) in df.index else np.nan
                    
                    target_row[feature_name] = val_a - val_b if not (np.isnan(val_a) or np.isnan(val_b)) else np.nan

        # Process the dataframes
        add_stats(row, form_df, "Form")
        add_stats(row, sos_df, "SOS")
        add_stats(row, ff_df, "FF")

        rows.append(row)

    return pd.DataFrame(rows)

# Create the training features
matchups = build_matchup_features(
    tourney, 
    elo_ratings, 
    seed_lookup, 
    form_df,    # Passed as DataFrame
    sos_df,     # Passed as DataFrame
    ff_df       # Passed as DataFrame
)

print(f'Matchup features: {matchups.shape}')
matchups.head()

Matchup features: (1717, 50)


,Season,TeamA,TeamB,Label,SeedA,SeedB,SeedDiff,EloA,EloB,EloDiff,Season_WinRate_Diff,Season_AvgMargin_Diff,Season_AvgOffEff_Diff,Season_AvgDefEff_Diff,Season_EffMargin_Diff,Season_AvgPace_Diff,Season_NumGames_Diff,Last10_WinRate_Diff,Last10_AvgMargin_Diff,Last10_AvgOffEff_Diff,Last10_AvgDefEff_Diff,Last10_EffMargin_Diff,Last10_AvgPace_Diff,Trend_EffMargin_Diff,Trend_WinRate_Diff,SOS_Season_AvgOppElo_Diff,SOS_Season_AvgOppEffMargin_Diff,SOS_Season_AvgOppWinRate_Diff,SOS_Last10_AvgOppElo_Diff,SOS_Last10_AvgOppEffMargin_Diff,SOS_Last10_AvgOppWinRate_Diff,Season_EFG_Diff,Season_EFGD_Diff,Season_TOR_Diff,Season_TORD_Diff,Season_ORB_Diff,Season_DRB_Diff,Season_FTR_Diff,Season_FTRD_Diff,Season_3PR_Diff,Season_3PRD_Diff,Season_3P_Diff,Season_3PD_Diff,Last10_EFG_Diff,Last10_EFGD_Diff,Last10_TOR_Diff,Last10_TORD_Diff,Last10_ORB_Diff,Last10_3PR_Diff,Last10_3P_Diff
0,1998,3104,3422,1,2,15,-13,1822.013865,1662.944715,159.069151,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.710402,NaN,0.117501,210.863802,NaN,0.223598,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1998,3112,3365,1,3,14,-11,1735.382681,1732.529838,2.852843,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.631462,NaN,0.093237,4.098475,NaN,-0.007774,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1998,3163,3193,1,2,15,-13,1879.762659,1653.284620,226.478040,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,106.858322,NaN,0.078057,124.759841,NaN,0.103994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1998,3198,3266,1,7,10,-3,1838.183532,1683.044642,155.138890,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-38.942056,NaN,-0.015033,-77.533338,NaN,-0.030486,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1998,3203,3208,1,10,7,3,1669.093495,1660.718139,8.375356,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-47.606679,NaN,-0.032827,-92.262747,NaN,-0.075686,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
def augment_matchups(df):
    """
    Duplicates every row in the matchup dataframe, swapping Team A and Team B,
    flipping the label, and negating all 'Diff' columns to create a perfectly 
    balanced dataset.
    """    
    # Create a copy for the flipped rows
    swapped_df = df.copy()
    
    # 1. Flip the Label (1 becomes 0, 0 becomes 1)
    swapped_df['Label'] = 1 - swapped_df['Label']
    
    # 2. Dynamically find all A and B column pairs
    # This looks for 'TeamA', 'EloA', 'Coach_TourneyWins_A', etc.
    cols_A = [col for col in df.columns if col.endswith('A')]
    cols_B = [col[:-1] + 'B' for col in cols_A]
    
    # Filter to only keep pairs where both A and B versions actually exist
    valid_pairs = [(a, b) for a, b in zip(cols_A, cols_B) if b in df.columns]
    valid_A = [a for a, b in valid_pairs]
    valid_B = [b for a, b in valid_pairs]
    
    # Swap the data between the A columns and B columns
    swapped_df[valid_A + valid_B] = df[valid_B + valid_A].values
    
    # 3. Dynamically find and negate all Difference columns
    # If Diff was (A - B), from the new perspective it must be (B - A), which is -(A - B)
    diff_cols = [col for col in df.columns if 'Diff' in col]
    swapped_df[diff_cols] = -swapped_df[diff_cols]
    
    # 4. Combine the original and swapped dataframes
    combined_df = pd.concat([df, swapped_df], axis=0, ignore_index=True)
    
    # 5. Shuffle the dataset so 1s and 0s are mixed (good for CV and batch training)
    combined_df = combined_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    
    return combined_df

# --- How to use it ---
# matchups = build_matchup_features(...)
print(f"Before augmentation: {matchups.shape[0]} rows, Label mean: {matchups['Label'].mean():.3f}")
balanced_matchups = augment_matchups(matchups)
print(f"After augmentation: {balanced_matchups.shape[0]} rows, Label mean: {balanced_matchups['Label'].mean():.3f}")

Before augmentation: 1717 rows, Label mean: 0.517
After augmentation: 3434 rows, Label mean: 0.500


## Walk Forward, Leave One Out, Cross Validation

In [15]:
def walk_forward_cv(matchups_df, feature_cols, val_start=VAL_START_SEASON,
                    impute=True, season_decay=None):
    SKIP_SEASONS = {2020}
    seasons     = sorted(matchups_df['Season'].unique())
    val_seasons = [s for s in seasons if s >= val_start and s not in SKIP_SEASONS]

    folds = []
    for val_season in val_seasons:
        train_raw = matchups_df[matchups_df['Season'] <  val_season]
        train = augment_matchups(train_raw) 
        val   = matchups_df[matchups_df['Season'] == val_season]

        if len(train) == 0 or len(val) == 0:
            continue

        y_train = train['Label']
        y_val   = val['Label']

        if impute:
            train_median = train[feature_cols].median()
            X_train = train[feature_cols].fillna(train_median)
            X_val   = val[feature_cols].fillna(train_median)
        else:
            X_train = train[feature_cols]
            X_val   = val[feature_cols]

        # Sample weights — exponential decay by season
        if season_decay is not None:
            max_season   = train['Season'].max()
            sample_weights = season_decay ** (max_season - train['Season'])
            sample_weights = sample_weights / sample_weights.sum() * len(train)
        else:
            sample_weights = None

        folds.append((val_season, X_train, y_train, X_val, y_val, sample_weights))

    return folds

## Compute Metrics

In [16]:
def compute_metrics(y_true, y_pred_proba):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred_proba)

    return {
        'brier_score': brier_score_loss(y_true, y_pred),
        'log_loss':    log_loss(y_true, y_pred),
        'roc_auc':     roc_auc_score(y_true, y_pred),
        'accuracy':    accuracy_score(y_true, (y_pred >= 0.5).astype(int)),
        'mean_pred':   y_pred.mean(),
        'n_games':     len(y_true),
    }


def plot_calibration(all_y_true, all_y_pred, run_name):
    fig, ax = plt.subplots(figsize=(6, 5))
    CalibrationDisplay.from_predictions(
        np.concatenate(all_y_true),
        np.concatenate(all_y_pred),
        n_bins=10, ax=ax, name=run_name
    )
    ax.set_title(f'Calibration — {run_name}')
    plt.tight_layout()
    return fig

## Run Experiment

In [23]:
def run_experiment(run_name, model, feature_cols, matchups_df,
                   tags=None, val_start=VAL_START_SEASON, scale=True, impute=True, season_decay=None):
    steps = []
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    pipeline = Pipeline(steps)

    folds = walk_forward_cv(matchups_df, feature_cols, val_start=val_start, impute=impute)

    fold_metrics = []
    all_y_true, all_y_pred = [], []

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param('features',         feature_cols)
        mlflow.log_param('n_features',       len(feature_cols))
        mlflow.log_param('model_type',       type(model).__name__)
        mlflow.log_param('val_start_season', val_start)
        mlflow.log_param('n_folds',          len(folds))
        mlflow.log_param('gender',           GENDER)
        mlflow.log_param('scale',            scale)

        if hasattr(model, 'get_params'):
            mlflow.log_params({
                f'model__{k}': v for k, v in model.get_params().items()
            })
        if tags:
            mlflow.set_tags(tags)

        for val_season, X_train, y_train, X_val, y_val, sample_weights in folds:
            fit_params = {}
            if sample_weights is not None:
                fit_params['model__sample_weight'] = sample_weights

            pipeline.fit(X_train, y_train, **fit_params)
            y_pred = pipeline.predict_proba(X_val)[:, 1]
            
            metrics = compute_metrics(y_val, y_pred)
            metrics['season'] = val_season
            fold_metrics.append(metrics)
            all_y_true.append(y_val.values)
            all_y_pred.append(y_pred)

            mlflow.log_metrics({
                f'fold_{val_season}_brier':    metrics['brier_score'],
                f'fold_{val_season}_logloss':  metrics['log_loss'],
                f'fold_{val_season}_auc':      metrics['roc_auc'],
                f'fold_{val_season}_accuracy': metrics['accuracy'],
            })

        metrics_df = pd.DataFrame(fold_metrics)
        agg = {
            'mean_brier':    metrics_df['brier_score'].mean(),
            'std_brier':     metrics_df['brier_score'].std(),
            'min_brier':     metrics_df['brier_score'].min(),
            'max_brier':     metrics_df['brier_score'].max(),
            'mean_logloss':  metrics_df['log_loss'].mean(),
            'mean_auc':      metrics_df['roc_auc'].mean(),
            'mean_accuracy': metrics_df['accuracy'].mean(),
        }
        modern_folds = metrics_df[metrics_df['season'] >= 2021]
        agg['modern_mean_brier'] = modern_folds['brier_score'].mean()
        agg['modern_std_brier']  = modern_folds['brier_score'].std()
        agg['modern_mean_auc']   = modern_folds['roc_auc'].mean()
        mlflow.log_metrics(agg)

        fig = plot_calibration(all_y_true, all_y_pred, run_name)
        mlflow.log_figure(fig, 'calibration_curve.png')
        plt.close(fig)

        final_train_df = augment_matchups(matchups_df.copy())
        
        if impute:
            X_all = final_train_df[feature_cols].fillna(final_train_df[feature_cols].median())
        else:
            X_all = final_train_df[feature_cols]
            
        y_all = final_train_df['Label']
        
        final_fit_params = {}
        
        if season_decay is not None:
            max_season = final_train_df['Season'].max()
            
            fresh_weights = season_decay ** (max_season - final_train_df['Season'])
            fresh_weights = fresh_weights / fresh_weights.mean()
            
            final_fit_params['model__sample_weight'] = fresh_weights

        pipeline.fit(X_all, y_all, **final_fit_params)
        mlflow.sklearn.log_model(pipeline, 'model')

    print(f'[{run_name}]')
    print(f'  Mean Brier (2016–2023) : {agg["mean_brier"]:.4f} ± {agg["std_brier"]:.4f}')
    print(f'  Mean Brier (2021–2023) : {agg["modern_mean_brier"]:.4f} ± {agg["modern_std_brier"]:.4f}')
    print(f'  Mean LogLoss           : {agg["mean_logloss"]:.4f}')
    print(f'  Mean AUC               : {agg["mean_auc"]:.4f}')
    print(f'  Mean Acc               : {agg["mean_accuracy"]:.4f}')
    print()

    return metrics_df, pipeline

## Model, Feature Sweep

In [ ]:
import itertools
import random
import lightgbm as lgb
import xgboost as xgb
import pandas as pd
from sklearn.metrics import brier_score_loss

# 1. Define the Feature Sets
base_features = [
    'SeedA', 'SeedB', 'SeedDiff', 
    'EloA', 'EloB', 'EloDiff', 
]

sos_features = [
    'SOS_Season_AvgOppElo_Diff', 'SOS_Season_AvgOppEffMargin_Diff', 'SOS_Season_AvgOppWinRate_Diff',
    'SOS_Last10_AvgOppElo_Diff', 'SOS_Last10_AvgOppEffMargin_Diff', 'SOS_Last10_AvgOppWinRate_Diff'
]

season_form = [
    'Season_WinRate_Diff', 'Season_AvgMargin_Diff', 'Season_AvgOffEff_Diff', 
    'Season_AvgDefEff_Diff', 'Season_EffMargin_Diff', 'Season_AvgPace_Diff'
]

last_10_trend = [
    'Last10_WinRate_Diff', 'Last10_AvgMargin_Diff', 'Last10_AvgOffEff_Diff', 
    'Last10_AvgDefEff_Diff', 'Last10_EffMargin_Diff', 'Last10_AvgPace_Diff',
    'Trend_EffMargin_Diff', 'Trend_WinRate_Diff'
]

four_factors = [
    'Season_EFG_Diff', 'Season_EFGD_Diff', 'Season_TOR_Diff', 'Season_TORD_Diff', 
    'Season_ORB_Diff', 'Season_DRB_Diff', 'Season_FTR_Diff', 'Season_FTRD_Diff', 
    'Season_3PR_Diff', 'Season_3PRD_Diff', 'Season_3P_Diff', 'Season_3PD_Diff'
]

optional_groups = {
    "SOS": sos_features,
    "Form": season_form,
    "Trend": last_10_trend,
    "FourFactors": four_factors
}

# 2. Build the 32 Combinations
all_combos = {"Baseline_Core_Only": base_features} 
group_names = list(optional_groups.keys())
for r in range(1, len(group_names) + 1):
    for combo_names in itertools.combinations(group_names, r):
        combo_name = "Core_+_" + "_+_".join(combo_names)
        combo_features = list(base_features) 
        for name in combo_names:
            combo_features.extend(optional_groups[name])
        all_combos[combo_name] = combo_features

# 3. Define Parameter Distributions
lgbm_param_dist = {
    'learning_rate': [0.01, 0.03, 0.05, 0.08],
    'max_depth': [3, 4, 5, 6],
    'num_leaves': [7, 15, 31, 63],
    'min_child_samples': [10, 30, 50, 100],  
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0], 
    'subsample': [0.7, 0.8, 0.9, 1.0]         
}

xgb_param_dist = {
    'learning_rate': [0.01, 0.03, 0.05, 0.08],
    'max_depth': [3, 4, 5, 6],
    'min_child_weight': [1, 3, 5, 7],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0], 
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

# 4. Split Data into CV (tuning) and Holdout (testing)
cv_matchups = matchups[matchups['Season'] <= 2023].copy()
holdout_matchups = matchups[matchups['Season'] >= 2024].copy()

# 5. Execute the Dual-Sweep
exhaustive_results = []
models_to_test = ['LightGBM', 'XGBoost']

print("Starting exhaustive sweep with strict holdout verification...\n")

for combo_name, features in all_combos.items():
    print(f"Testing: {combo_name} ({len(features)} features)")
    
    for model_type in models_to_test:
        best_cv_brier = float('inf')
        best_params = {}
        
        # 5a. Hyperparameter Search Loop
        for i in range(10): 
            if model_type == 'LightGBM':
                params = {
                    'n_estimators': 150,
                    'learning_rate': random.choice(lgbm_param_dist['learning_rate']),
                    'max_depth': random.choice(lgbm_param_dist['max_depth']),
                    'min_child_samples': random.choice(lgbm_param_dist['min_child_samples']),
                    'colsample_bytree': random.choice(lgbm_param_dist['colsample_bytree']),
                    'subsample': random.choice(lgbm_param_dist['subsample']),
                    'random_state': 42 + i,
                    'verbose': -1
                }
                max_possible_leaves = 2 ** params['max_depth'] - 1
                params['num_leaves'] = min(random.choice(lgbm_param_dist['num_leaves']), max_possible_leaves)
                model = lgb.LGBMClassifier(**params)
                
            else: # XGBoost
                params = {
                    'n_estimators': 150,
                    'learning_rate': random.choice(xgb_param_dist['learning_rate']),
                    'max_depth': random.choice(xgb_param_dist['max_depth']),
                    'min_child_weight': random.choice(xgb_param_dist['min_child_weight']),
                    'colsample_bytree': random.choice(xgb_param_dist['colsample_bytree']),
                    'subsample': random.choice(xgb_param_dist['subsample']),
                    'random_state': 42 + i,
                    'eval_metric': 'logloss'
                }
                model = xgb.XGBClassifier(**params)
            
            # Run Walk-Forward CV on the restricted dataset (up to 2023)
            short_params = f"LR{params['learning_rate']}_D{params['max_depth']}_F{params['colsample_bytree']}"

            # 2. Combine it with the model type (and optionally the combo_name)
            current_run_name = f"{model_type}_{short_params}"
            metrics_df, _ = run_experiment(
                run_name=current_run_name, 
                model=model,
                feature_cols=features,
                matchups_df=cv_matchups,
                val_start=2016,
                scale=False,
                impute=False
            )
            
            cv_brier = metrics_df['brier_score'].mean()
            
            if cv_brier < best_cv_brier:
                best_cv_brier = cv_brier
                best_params = params.copy()
                
        # 5b. Holdout Verification
        # Retrain a model with the best parameters on the full augmented CV set
        if model_type == 'LightGBM':
            final_cv_model = lgb.LGBMClassifier(**best_params)
        else:
            final_cv_model = xgb.XGBClassifier(**best_params)
            
        augmented_cv = augment_matchups(cv_matchups.copy())
        X_train_cv = augmented_cv[features]
        y_train_cv = augmented_cv['Label']
        final_cv_model.fit(X_train_cv, y_train_cv)
        
        # Predict on the untouched 2024-2025 holdout set
        X_holdout = holdout_matchups[features]
        y_holdout = holdout_matchups['Label']
        preds_holdout = final_cv_model.predict_proba(X_holdout)[:, 1]
        holdout_brier = brier_score_loss(y_holdout, preds_holdout)
        
        param_str = f"LR:{best_params['learning_rate']}, Depth:{best_params['max_depth']}, FeatFrac:{best_params['colsample_bytree']}"
        
        exhaustive_results.append({
            'Model': model_type,
            'Combo': combo_name,
            'CV_Brier': best_cv_brier,
            'Holdout_Brier': holdout_brier,
            'Num_Features': len(features),
            'Best_Params': param_str
        })

# 6. Output the Results
print("\n--- EXHAUSTIVE SWEEP COMPLETE ---")
results_df = pd.DataFrame(exhaustive_results)
# Sort by Holdout Brier to identify the models that generalize best
results_df = results_df.sort_values('Holdout_Brier')
print(results_df.head(15).to_string(index=False))

Starting exhaustive sweep with strict holdout verification...

Testing: Baseline_Core_Only (6 features)
[LightGBM_LR0.03_D5_F0.7]
  Mean Brier (2016–2023) : 0.1509 ± 0.0209
  Mean Brier (2021–2023) : 0.1576 ± 0.0100
  Mean LogLoss           : 0.4443
  Mean AUC               : 0.8668
  Mean Acc               : 0.7488

[LightGBM_LR0.01_D4_F0.8]
  Mean Brier (2016–2023) : 0.1532 ± 0.0175
  Mean Brier (2021–2023) : 0.1585 ± 0.0109
  Mean LogLoss           : 0.4704
  Mean AUC               : 0.8670
  Mean Acc               : 0.7641

[LightGBM_LR0.08_D3_F0.7]
  Mean Brier (2016–2023) : 0.1529 ± 0.0223
  Mean Brier (2021–2023) : 0.1588 ± 0.0137
  Mean LogLoss           : 0.4500
  Mean AUC               : 0.8602
  Mean Acc               : 0.7534

[LightGBM_LR0.01_D3_F1.0]
  Mean Brier (2016–2023) : 0.1546 ± 0.0173
  Mean Brier (2021–2023) : 0.1598 ± 0.0129
  Mean LogLoss           : 0.4754
  Mean AUC               : 0.8656
  Mean Acc               : 0.7617

[LightGBM_LR0.08_D4_F1.0]
  Mean Bri

## Ready for Submission

In [ ]:
print("Initializing Final 2026 Women's Ensemble...")

# 1. Define the Winning Women's Feature Sets from your Sweep
# Winner: Core + SOS + Form (18 features)
features_xgb_w_heavy = base_features + sos_features + season_form 

# Second Best: Core + SOS (12 features)
features_lgbm_w_robust = base_features + sos_features

# Base Anchor
features_anchor_w = base_features

# Note: We ensure matchups_w is used here (the historical Women's data)
full_train_df_w = augment_matchups(matchups.copy())

print("Training Model 1 (XGBoost: Core + SOS + Form)")
# Hyperparams: LR: 0.03, Depth: 4, FeatFrac: 0.8
model_xgb_w = xgb.XGBClassifier(
    n_estimators=150,      # Slightly higher estimators for lower learning rate
    learning_rate=0.03,    # UPDATED
    max_depth=4,           # UPDATED
    colsample_bytree=0.8,  # UPDATED
    subsample=0.8, 
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
model_xgb_w.fit(full_train_df_w[features_xgb_w_heavy], full_train_df_w['Label'])

print("Training Model 2 (LightGBM: Core + SOS)")
# Hyperparams: LR: 0.03, Depth: 5, FeatFrac: 0.7
model_lgbm_w = lgb.LGBMClassifier(
    n_estimators=150, 
    learning_rate=0.03,    # UPDATED
    max_depth=5,           # UPDATED
    colsample_bytree=0.7,  # UPDATED
    subsample=0.7, 
    random_state=42, 
    verbose=-1
)
model_lgbm_w.fit(full_train_df_w[features_lgbm_w_robust], full_train_df_w['Label'])

print("Training Model 3 (LightGBM: Baseline Anchor)")
# Hyperparams: LR: 0.05, Depth: 6, FeatFrac: 0.6
model_anchor_w = lgb.LGBMClassifier(
    n_estimators=150, 
    learning_rate=0.05, 
    max_depth=6, 
    colsample_bytree=0.6, 
    random_state=42, 
    verbose=-1
)
model_anchor_w.fit(full_train_df_w[features_anchor_w], full_train_df_w['Label'])

print("Processing 2026 Women's Matchups...")
# Filter sub_df for Women's TeamIDs (3000-3999)
sub_df = pd.read_csv("../data/raw/SampleSubmissionStage2.csv")
sub_df['Season'] = sub_df['ID'].apply(lambda x: int(x.split('_')[0]))
sub_df['WTeamID'] = sub_df['ID'].apply(lambda x: int(x.split('_')[1]))
sub_df['LTeamID'] = sub_df['ID'].apply(lambda x: int(x.split('_')[2]))
womens_sub = sub_df[sub_df['WTeamID'] >= 3000].copy()

# Build features using Women's specific DataFrames (elo_ratings_w, etc.)
sub_features_df_w = build_matchup_features(
    womens_sub, elo_ratings, seed_lookup, 
    form_df, sos_df, ff_df
)

print("🔮 Generating Ensemble Predictions...")
pred_xgb_w = model_xgb_w.predict_proba(sub_features_df_w[features_xgb_w_heavy])[:, 1]
pred_lgbm_w = model_lgbm_w.predict_proba(sub_features_df_w[features_lgbm_w_robust])[:, 1]
pred_anchor_w = model_anchor_w.predict_proba(sub_features_df_w[features_anchor_w])[:, 1]

# Weighting to favor your 0.1119 Brier winner (XGBoost)
womens_sub['Pred'] = (pred_xgb_w * 0.4) + (pred_lgbm_w * 0.3) + (pred_anchor_w * 0.3)

final_sub_w = womens_sub[['ID', 'Pred']].copy()
final_sub_w.to_csv("submission_womens_2026.csv", index=False)

print(f"\n🎉 SUCCESS! Generated {len(final_sub_w)} Women's predictions.")

Initializing Final 2026 Women's Ensemble...
Training Model 1 (XGBoost: Core + SOS + Form)
Training Model 2 (LightGBM: Core + SOS)
Training Model 3 (LightGBM: Baseline Anchor)
Processing 2026 Women's Matchups...
🔮 Generating Ensemble Predictions...

🎉 SUCCESS! Generated 65703 Women's predictions.


## Simulation

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

prob_lookup = final_sub_w.set_index('ID')['Pred'].to_dict()

# Load mapping files
slots = pd.read_csv("../data/raw/WNCAATourneySlots.csv")
seeds = pd.read_csv("../data/raw/WNCAATourneySeeds.csv")
slots_2026 = slots[slots['Season'] == 2026].copy()

# Base seeds for 2026
seeds_2026_base = seeds[seeds['Season'] == 2026].set_index('Seed')['TeamID'].to_dict()

prob_lookup = final_sub_w.set_index('ID')['Pred'].to_dict()

def simulate_tournament(slots_df, seeds_dict, p_lookup):
    results = {}
    # We create a local copy of seeds to hold play-in winners
    curr_seeds = seeds_dict.copy()
    
    # --- PHASE 1: Resolve First Four (Play-In Slots) ---
    # These are slots like 'X10' that depend on 'X10a' and 'X10b'
    # Based on your data: X10, X16, Y16, Z11
    play_in_slots = slots_df[slots_df['StrongSeed'].str.contains('[ab]', na=False)]['Slot'].tolist()
    
    for slot_name in play_in_slots:
        slot_info = slots_df[slots_df['Slot'] == slot_name].iloc[0]
        
        t_a = curr_seeds[slot_info['StrongSeed']]
        t_b = curr_seeds[slot_info['WeakSeed']]
        
        # Get Model Probability
        low, high = min(t_a, t_b), max(t_a, t_b)
        match_id = f"2026_{low}_{high}"
        
        prob_a = p_lookup.get(match_id, 0.5) # Default to 0.5 if matchup missing
        if t_a != low: prob_a = 1 - prob_a
        
        # Store winner in curr_seeds so R1 can find it
        winner = t_a if np.random.random() < prob_a else t_b
        curr_seeds[slot_name] = winner

    # --- PHASE 2: Main Rounds (R1 through R6) ---
    # We iterate through rounds 1 to 6
    for r in range(1, 7):
        round_slots = slots_df[slots_df['Slot'].str.startswith(f'R{r}')]
        
        for _, slot in round_slots.iterrows():
            slot_id = slot['Slot']
            
            # For Round 1, participants come from curr_seeds
            # For Rounds 2-6, participants come from results
            if r == 1:
                t_a = curr_seeds[slot['StrongSeed']]
                t_b = curr_seeds[slot['WeakSeed']]
            else:
                t_a = results[slot['StrongSeed']]
                t_b = results[slot['WeakSeed']]
            
            low, high = min(t_a, t_b), max(t_a, t_b)
            match_id = f"2026_{low}_{high}"
            
            prob_a = p_lookup.get(match_id, 0.5)
            if t_a != low: prob_a = 1 - prob_a
            
            results[slot_id] = t_a if np.random.random() < prob_a else t_b
            
    return tuple(sorted(results.items()))

# --- EXECUTION ---
print("🎲 Simulating 10,000 Women's Tournaments...")
all_brackets = [simulate_tournament(slots_2026, seeds_2026_base, prob_lookup) for _ in range(10000)]

bracket_counts = Counter(all_brackets)
top_5 = bracket_counts.most_common(5)

🎲 Simulating 10,000 Women's Tournaments...

🏆 Simulation Complete.
Bracket 1 (1 times): Winner ID 3376
Bracket 2 (1 times): Winner ID 3163
Bracket 3 (1 times): Winner ID 3376
Bracket 4 (1 times): Winner ID 3400
Bracket 5 (1 times): Winner ID 3326


In [32]:
from collections import defaultdict

# 1. Count wins per team across all 10k brackets
team_win_counts = defaultdict(int)

for bracket in all_brackets:
    # Each 'bracket' is a tuple of (Slot, WinnerID)
    # We count how many slots a team won in this specific simulation
    for slot, winner_id in bracket:
        team_win_counts[winner_id] += 1

# 2. Convert to Expected Wins (EV)
# Load Women's names (Crucial fix!)
w_teams = pd.read_csv("../data/raw/WTeams.csv")
w_names = w_teams.set_index('TeamID')['TeamName'].to_dict()

ev_list = []
for tid, wins in team_win_counts.items():
    ev_list.append({
        'Team': w_names.get(tid, f"Unknown_{tid}"),
        'EV': wins / 10000
    })

# 3. Display Top 15
ev_df = pd.DataFrame(ev_list).sort_values(by='EV', ascending=False)
print("\n🏆 --- WOMEN'S TOURNAMENT: TOP 15 BY EXPECTED WINS (EV) ---")
print(ev_df.head(15).to_string(index=False))


🏆 --- WOMEN'S TOURNAMENT: TOP 15 BY EXPECTED WINS (EV) ---
          Team     EV
   Connecticut 4.5519
          UCLA 4.2623
         Texas 4.1211
South Carolina 4.1185
           LSU 2.8513
           TCU 2.4827
    Louisville 2.4317
          Iowa 2.4176
    Vanderbilt 2.2907
          Duke 2.2766
      Michigan 2.2716
       Ohio St 2.0233
 West Virginia 1.5984
      Oklahoma 1.5961
     Minnesota 1.5789


In [35]:
def print_final_chalk_bracket(sub_df, slots_df, seeds_df, team_names):
    # 1. Setup Initial Seeds (includes a/b play-ins)
    seeds_2026 = seeds_df[seeds_df['Season'] == 2026].set_index('Seed')['TeamID'].to_dict()
    bracket_results = {}
    
    # 2. PHASE 0: Resolve Play-In Slots (X10, X16, Y16, Z11)
    # These create the base seeds for Round 1
    play_in_slots = ['X10', 'X16', 'Y16', 'Z11']
    print("--- FIRST FOUR (PLAY-INS) ---")
    
    for slot_id in play_in_slots:
        slot = slots_df[slots_df['Slot'] == slot_id].iloc[0]
        t_a, t_b = seeds_2026[slot['StrongSeed']], seeds_2026[slot['WeakSeed']]
        
        low, high = min(t_a, t_b), max(t_a, t_b)
        match_id = f"2026_{low}_{high}"
        prob_low = sub_df.loc[sub_df['ID'] == match_id, 'Pred'].values[0]
        prob_a = prob_low if t_a == low else (1 - prob_low)
        
        winner = t_a if prob_a >= 0.5 else t_b
        # THIS IS THE FIX: Put the winner back into seeds_2026 as the main seed
        seeds_2026[slot_id] = winner
        print(f"{slot_id} Play-in: {team_names[t_a]} vs {team_names[t_b]} -> **{team_names[winner]}**")

    # 3. PHASE 1-6: Main Tournament
    for r in range(1, 7):
        print(f"\n--- ROUND {r} ---")
        round_slots = slots_df[slots_df['Slot'].str.startswith(f'R{r}')]
        
        for _, slot in round_slots.iterrows():
            slot_id = slot['Slot']
            
            # Round 1 pulls from the updated seeds_2026 (which now has X16, etc.)
            # Rounds 2-6 pull from bracket_results
            t_a = seeds_2026[slot['StrongSeed']] if r == 1 else bracket_results[slot['StrongSeed']]
            t_b = seeds_2026[slot['WeakSeed']] if r == 1 else bracket_results[slot['WeakSeed']]
            
            low, high = min(t_a, t_b), max(t_a, t_b)
            match_id = f"2026_{low}_{high}"
            prob_low = sub_df.loc[sub_df['ID'] == match_id, 'Pred'].values[0]
            prob_a = prob_low if t_a == low else (1 - prob_low)
            
            winner = t_a if prob_a >= 0.5 else t_b
            bracket_results[slot_id] = winner
            
            print(f"{slot_id}: {team_names[t_a]} vs {team_names[t_b]} -> **{team_names[winner]}** ({prob_a:.1%})")

print_final_chalk_bracket(final_sub_w, slots_2026, seeds, w_names)

--- FIRST FOUR (PLAY-INS) ---
X10 Play-in: Arizona St vs Virginia -> **Virginia**
X16 Play-in: Samford vs Southern Univ -> **Southern Univ**
Y16 Play-in: Missouri St vs SF Austin -> **Missouri St**
Z11 Play-in: Nebraska vs Richmond -> **Nebraska**

--- ROUND 1 ---
R1W1: Connecticut vs UT San Antonio -> **Connecticut** (98.7%)
R1W2: Vanderbilt vs High Point -> **Vanderbilt** (98.8%)
R1W3: Ohio St vs Howard -> **Ohio St** (98.9%)
R1W4: North Carolina vs W Illinois -> **North Carolina** (97.4%)
R1W5: Maryland vs Murray St -> **Maryland** (89.0%)
R1W6: Notre Dame vs Fairfield -> **Notre Dame** (66.9%)
R1W7: Illinois vs Colorado -> **Illinois** (54.1%)
R1W8: Iowa St vs Syracuse -> **Iowa St** (61.4%)
R1X1: South Carolina vs Southern Univ -> **South Carolina** (99.2%)
R1X2: Iowa vs F Dickinson -> **Iowa** (98.8%)
R1X3: TCU vs UC San Diego -> **TCU** (99.0%)
R1X4: Oklahoma vs Idaho -> **Oklahoma** (95.4%)
R1X5: Michigan St vs Colorado St -> **Michigan St** (79.9%)
R1X6: Washington vs S Dakota

In [36]:
from collections import defaultdict
import pandas as pd

# 1. Load Women's Team Names (Women's IDs are 3000+)
w_teams = pd.read_csv("../data/raw/WTeams.csv")
team_names = w_teams.set_index('TeamID')['TeamName'].to_dict()

# round_reach[team_id][round_name] = count
round_reach = defaultdict(lambda: defaultdict(int))

# 2. Track how often each team reaches each round
# Note: Women's slots in 2026 still follow R1, R2, etc.
for bracket in all_brackets:
    for slot, winner in bracket:
        round_prefix = slot[:2] # 'R1', 'R2', 'R3', 'R4', 'R5', 'R6'
        round_reach[winner][round_prefix] += 1

# Total simulations used for percentages
total_sims = 10000

# 3. Print most frequent Champions (R6)
print("🏆 --- WOMEN'S NATIONAL CHAMPIONS (2026) ---")
champ_counts = []
for team_id, rounds in round_reach.items():
    if 'R6' in rounds:
        champ_counts.append((team_names.get(team_id, f"ID_{team_id}"), rounds['R6']))

champ_counts.sort(key=lambda x: x[1], reverse=True)
for name, count in champ_counts[:10]:
    print(f"{name:.<20} {count:>6} wins ({count/total_sims:.2%})")

# 4. Print Final Four appearances (R4 winners)
# R4 determines the winners of Fort Worth 1, Sacramento 2, Fort Worth 3, and Sacramento 4
print("\n🔥 --- WOMEN'S FINAL FOUR APPEARANCES (2026) ---")
f4_counts = []
for team_id, rounds in round_reach.items():
    if 'R4' in rounds:
        f4_counts.append((team_names.get(team_id, f"ID_{team_id}"), rounds['R4']))

f4_counts.sort(key=lambda x: x[1], reverse=True)
for name, count in f4_counts[:10]:
    print(f"{name:.<20} {count:>6} apps ({count/total_sims:.2%})")

🏆 --- WOMEN'S NATIONAL CHAMPIONS (2026) ---
Connecticut.........   3713 wins (37.13%)
UCLA................   2326 wins (23.26%)
South Carolina......   1781 wins (17.81%)
Texas...............   1572 wins (15.72%)
LSU.................    347 wins (3.47%)
TCU.................     48 wins (0.48%)
Michigan............     45 wins (0.45%)
Duke................     33 wins (0.33%)
Louisville..........     32 wins (0.32%)
Vanderbilt..........     28 wins (0.28%)

🔥 --- WOMEN'S FINAL FOUR APPEARANCES (2026) ---
Connecticut.........   7906 apps (79.06%)
South Carolina......   7692 apps (76.92%)
Texas...............   7506 apps (75.06%)
UCLA................   7092 apps (70.92%)
LSU.................   2080 apps (20.80%)
TCU.................   1053 apps (10.53%)
Michigan............   1030 apps (10.30%)
Louisville..........    886 apps (8.86%)
Iowa................    827 apps (8.27%)
Vanderbilt..........    715 apps (7.15%)
